In [2]:
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.normalizers import NFKC
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer

# 1. Stream 500 samples from FineWeb
dataset = load_dataset(
    "HuggingFaceFW/fineweb",
    name="sample-10BT",
    split="train",
    streaming=True,
)
samples = dataset.take(500)


# Generator to yield text batches efficiently
def get_training_corpus():
    for sample in samples:
        yield sample["text"]


# 2. Initialize a Byte-Level BPE tokenizer (similar to GPT-2 / Llama)
tokenizer = Tokenizer(BPE(unk_token="<|unk|>"))
tokenizer.normalizer = NFKC()
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

# 3. Configure the trainer
trainer = BpeTrainer(
    vocab_size=8000,  # Keep small (e.g., 4k–8k) since 500 samples is a small corpus
    special_tokens=["<|endoftext|>", "<|unk|>", "<|pad|>"],
    initial_alphabet=ByteLevel.alphabet(),
)

# 4. Train and save the tokenizer
tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)
tokenizer.save("fineweb_tokenizer.json")

# 5. Test it
encoded = tokenizer.encode("Hello, testing my custom FineWeb tokenizer!")
print("Tokens:", encoded.tokens)
print("IDs:", encoded.ids)

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Tokens: ['H', 'ell', 'o', ',', 'Ġtesting', 'Ġmy', 'Ġcustom', 'ĠF', 'ine', 'Web', 'Ġto', 'ken', 'iz', 'er', '!']
IDs: [42, 816, 81, 14, 5929, 628, 1066, 370, 473, 5381, 286, 4113, 551, 266, 3]
